In [2]:
import pandas as pd
import spacy
from spacy.matcher import PhraseMatcher
from skillNer.skill_extractor_class import SkillExtractor
from skillNer.general_params import SKILL_DB

# 1. Initialize the NLP models (Only run this cell once, it takes a moment to load)
nlp = spacy.load("en_core_web_sm")
skill_extractor = SkillExtractor(nlp, SKILL_DB, PhraseMatcher)

def extract_skills(text):
    if not isinstance(text, str):
        return []
    try:
        annotations = skill_extractor.annotate(text)
        extracted = set()
        
        for match in annotations.get("results", {}).get("full_matches", []):
            if name := match.get("doc_node_value"):
                extracted.add(name.lower())
                
        for match in annotations.get("results", {}).get("ngram_scored", []):
            if name := match.get("doc_node_value"):
                extracted.add(name.lower())
                
        return sorted(list(extracted))
    except Exception:
        return []

# 2. Load your dataset
df = pd.read_csv("LinkedIn_RDB_three.csv")

# 3. Take a random sample of 5 job postings to test
sample_df = df.sample(n=5, random_state=42).copy()

# 4. Apply the extraction (this might take a few seconds per row)
print("Extracting skills... please wait.\n")
sample_df['extracted_skills'] = sample_df['description'].apply(extract_skills)

# 5. Review the results
for index, row in sample_df.iterrows():
    print(f"Job Title: {row['title']}")
    print(f"Extracted Skills: {row['extracted_skills']}")
    print("-" * 50)

loading full_matcher ...
loading abv_matcher ...
loading full_uni_matcher ...
loading low_form_matcher ...
loading token_matcher ...
Extracting skills... please wait.



c:\Users\Deepak\AIHire\backend\venv\Lib\site-packages\skillNer\utils.py:99: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Token.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  vec_similarity = token1.similarity(token2)


Job Title: Senior Internal Events Manager 
Extracted Skills: ['communication plans', 'communications', 'copywriting', 'e', 'emotional intelligence', 'employee engagement', 'event management', 'event planning', 'events manager', 'internal communication', 'intranet', 'management partner', 'presentations', 'public relation', 'san', 'scale', 'slack', 'storytelling', 'track', 'work environments', 'xcelerator']
--------------------------------------------------
Job Title: Desktop Support Engineer
Extracted Skills: ['active directory', 'ad', 'dns', 'infrastructure networks', 'management azure', 'microsoft networks', 'modems', 'operating system', 'organizational structure', 'problem solve', 'san', 'self sufficiency', 'server virtualization', 'teamwork', 'thin client', 'track', 'zoom', 'zoom room']
--------------------------------------------------
Job Title: Assistant Manager
Extracted Skills: ['basic life', 'basic math', 'business card', 'cleanliness', 'closing', 'communication skills', 'comp

In [4]:
# 1. Your actual parsed resume text
resume_text = """
PROFESSIONAL SUMMARY
Machine Learning Engineer building production-ready models with Python, PyTorch, and
scikit-learn. Develops NLP, recommendation, and predictive features, then packages them
through FastAPI for deployment. Bridges model development, API design, and database
integration with PostgreSQL and SQLAlchemy.
SKILLS
Languages: Python, SQL, JavaScript
ML/DL/NLP: PyTorch, scikit-learn, NLP, Machine Learning, Deep Learning,
Recommendation Systems
Backend: FastAPI, REST APIs, SQLAlchemy
Database: PostgreSQL
Frontend: React, Vite
Tools: Git, GitHub
"""

# 2. Our slightly upgraded extractor function
def extract_skills_clean(text):
    if not isinstance(text, str):
        return []
    try:
        annotations = skill_extractor.annotate(text)
        extracted = set()
        
        for match in annotations.get("results", {}).get("full_matches", []):
            if name := match.get("doc_node_value"):
                if len(name) > 1:  # Filter out single-letter noise
                    extracted.add(name.lower())
                
        for match in annotations.get("results", {}).get("ngram_scored", []):
            if name := match.get("doc_node_value"):
                if len(name) > 1:  # Filter out single-letter noise
                    extracted.add(name.lower())
                
        return sorted(list(extracted))
    except Exception as e:
        print(f"Error: {e}")
        return []

# 3. Test it out
my_skills = extract_skills_clean(resume_text)
print("My Extracted Skills:")
print(my_skills)

My Extracted Skills:
['api design', 'backend', 'bridges', 'deep learning', 'git', 'github', 'integration', 'javascript', 'machine learn', 'model development', 'postgresql', 'python', 'pytorch', 'react', 'recommendation systems', 'rest apis', 'scikit learn', 'sql', 'sqlalchemy']
